# Gemini Logical `star_rz` demo

This notebook shows the first-class Gemini Logical `star_rz` statement. The kernel prepares one logical Steane block, applies a logical STAR/TMR `Rz(pi / 16)` injection layer, lowers it through the logical compiler, and then builds the noiseless `tsim` circuit so the physical circuit can be inspected.

In [1]:
import math

from kirin import rewrite

from bloqade import squin, tsim
from bloqade.gemini import logical as gemini_logical
from bloqade.lanes.arch.gemini.physical import get_arch_spec as get_physical_arch_spec
from bloqade.lanes.logical_mvp import compile_squin_to_move
from bloqade.lanes.noise_model import generate_logical_noise_model
from bloqade.lanes.rewrite.squin2stim import RemoveReturn
from bloqade.lanes.transform import MoveToSquinLogical

## Logical kernel

`star_rz(theta, q)` is a logical statement. The default support is the Steane logical-Z line `(4, 5, 6)`. You can pass another valid Steane line with `qubit_indices=...`.

In [2]:
theta = math.pi / 16


@gemini_logical.kernel(aggressive_unroll=True)
def star_kernel():
    reg = squin.qalloc(1)
    squin.u3(0.5, 0.5, 0.5, reg[0])
    gemini_logical.star_rz(theta, reg[0])
    gemini_logical.terminal_measure(reg)


star_kernel.print()

func.func @star_kernel() -> !py.NoneType {
  ^0(%star_kernel_self):
  │  %qubit = qubit.new() : !py.Qubit
  │    %reg = py.ilist.new(values=(%qubit)){elem_type=!Any} : !py.IList[!py.Qubit, Literal(1,int)]
  │ %qubits = py.ilist.new(values=(%qubit)){elem_type=!py.Qubit} : !py.IList[!py.Qubit, Literal(1,int)]
  │      %0 = py.constant.constant 0.07957747154594767 : !py.float
  │           squin.gate.u3(theta=%0, phi=%0, lam=%0, qubits=%qubits)
  │      %1 = py.constant.constant 0.19634954084936207 : !py.float
  │           logical.star_rz(rotation_angle=%1, qubits=%qubit){qubit_indices=(4, 5, 6) : !py.tuple}
  │      %2 = logical.terminal_logical_measurement(qubits=%reg){num_physical_qubits=7 : !py.int} : !py.IList[!py.IList[!py.MeasurementResult, !Any], Literal(1,int)]
  │      %3 = func.const.none() : !py.NoneType
  │           func.return %3
} // func.func star_kernel


## Logical Move kernel

With `transversal_rewrite=False`, the compiler lowers into the logical Move dialect and keeps the special `lanes.move.star_rz` statement.

In [3]:
logical_move_kernel = compile_squin_to_move(
    star_kernel,
    transversal_rewrite=False,
    no_raise=False,
    insert_return_moves=False,
)
logical_move_kernel.print()

func.func @star_kernel() -> !py.NoneType {
  ^0(%star_kernel_self):
  │ %0 = lanes.move.load() : !py.State
  │ %1 = lanes.move.fill(current_state=%0){location_addresses=(0x0000000000000000,) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │      lanes.move.store(current_state=%1)
  │ %2 = py.constant.constant 0.07957747154594767 : !py.float
  │ %3 = lanes.move.logical_initialize(current_state=%1, thetas=(%2), phis=(%2), lams=(%2)){location_addresses=(0x0000000000000000,) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │      lanes.move.store(current_state=%3)
  │ %4 = py.constant.constant 0.19634954084936207 : !py.float
  │ %5 = lanes.move.star_rz(current_state=%3, rotation_angle=%4){location_addresses=(0x0000000000000000,) : !py.tuple, qubit_indices=(4, 5, 6) : !py.tuple} : !py.State
  │      lanes.move.store(current_state=%5)
  │ %6 = lanes.move.end_measure(current_state=%5){zone_addresses=(0x0000000000000000,) : !py.tuple[!py.ZoneAddress, ...]} : !py.MeasurementFuture
  │ 

## Physical Move kernel

With `transversal_rewrite=True`, `lanes.move.star_rz` is rewritten before the generic transversal location rewrite. For `k = 3`, the compiler computes `theta_star` and emits one physical `lanes.move.local_rz` over physical sites `(4, 5, 6)`.

In [4]:
physical_move_kernel = compile_squin_to_move(
    star_kernel,
    transversal_rewrite=True,
    no_raise=False,
    insert_return_moves=False,
)
physical_move_kernel.print()

func.func @star_kernel() -> !py.NoneType {
  ^0(%star_kernel_self):
  │  %0 = lanes.move.load() : !py.State
  │  %1 = lanes.move.fill(current_state=%0){location_addresses=(0x0000000000000000, 0x0000000001000000, 0x0000000002000000, 0x0000000003000000, 0x0000000004000000, 0x0000000005000000, 0x0000000006000000) : !py.tuple[!py.LocationAddress, ...]} : !py.State
  │       lanes.move.store(current_state=%1)
  │  %2 = py.constant.constant 0.07957747154594767 : !py.float
  │  %3 = lanes.move.physical_initialize(current_state=%1, thetas=(%2), phis=(%2), lams=(%2)){location_addresses=((0x0000000000000000, 0x0000000001000000, 0x0000000002000000, 0x0000000003000000, 0x0000000004000000, 0x0000000005000000, 0x0000000006000000),) : !py.tuple[!py.tuple[!py.LocationAddress, ...], ...]} : !py.State
  │       lanes.move.store(current_state=%3)
  │  %4 = py.constant.constant 0.19634954084936207 : !py.float
  │  %5 = py.constant.constant -0.865268077594109 : !py.float
  │  %6 = lanes.move.local_rz(curre

## Physical Squin kernel

The physical Move kernel can now be lowered to a noiseless physical Squin kernel. This is the kernel that `tsim` consumes.

In [5]:
physical_arch = get_physical_arch_spec()
noise_model = generate_logical_noise_model()

physical_squin_kernel = MoveToSquinLogical(
    arch_spec=physical_arch,
    noise_model=noise_model,
    add_noise=False,
).emit(physical_move_kernel, no_raise=False)

physical_squin_kernel.print()

func.func @star_kernel() -> !py.NoneType {
  ^0(%star_kernel_self):
  │  %0 = qubit.new() : !py.Qubit
  │  %1 = qubit.new() : !py.Qubit
  │  %2 = qubit.new() : !py.Qubit
  │  %3 = qubit.new() : !py.Qubit
  │  %4 = qubit.new() : !py.Qubit
  │  %5 = qubit.new() : !py.Qubit
  │  %6 = qubit.new() : !py.Qubit
  │  %7 = py.constant.constant 0.5 : !py.float
  │  %8 = py.ilist.new(values=(%0, %1, %2, %3, %4, %5, %6)){elem_type=!py.Qubit} : !py.IList[!py.Qubit, Literal(7,int)]
  │  %9 = func.invoke steane7_initialize(%7, %7, %7, %8) : !py.NoneType maybe_pure=False
  │ %10 = py.constant.constant -0.865268077594109 : !py.float
  │ %11 = py.constant.constant 0.0 : !py.float
  │ %12 = py.ilist.new(values=(%4, %5, %6)){elem_type=!py.Qubit} : !py.IList[!py.Qubit, Literal(3,int)]
  │       squin.gate.u3(theta=%11, phi=%10, lam=%11, qubits=%12)
  │ %13 = func.invoke measure(%0) : !py.MeasurementResult maybe_pure=False
  │ %14 = func.invoke measure(%1) : !py.MeasurementResult maybe_pure=False
  │ %15 = 

## `tsim` circuit

The `RemoveReturn` rewrite strips the final Kirin return statement before constructing the `tsim` circuit. The diagram below shows the physical circuit after the full STAR lowering path.

In [6]:
tsim_kernel = physical_squin_kernel.similar()
rewrite.Walk(RemoveReturn()).rewrite(tsim_kernel.code)

tsim_circuit = tsim.Circuit(tsim_kernel)
print(f"num_qubits = {tsim_circuit.num_qubits}")
print(f"is_clifford = {tsim_circuit.is_clifford}")

tsim_circuit.diagram(width=900, height=tsim_circuit.num_qubits * 45)

num_qubits = 7
is_clifford = False


# Using GeminiLogicalSimulator

In [7]:
from bloqade.gemini.device import GeminiLogicalSimulator

In [8]:
task = GeminiLogicalSimulator().task(star_kernel)
task.noiseless_tsim_circuit.diagram(width=900)